In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

cwd = Path.cwd()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd

PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

news_all = pd.read_csv(PROCESSED_DIR / "news_all.csv")
dev_samples = pd.read_csv(PROCESSED_DIR / "dev_samples_5000.csv")

news_all["text"] = news_all["text"].fillna("")
dev_samples["history"] = dev_samples["history"].fillna("")

print("news_all:", news_all.shape)
print("dev_samples:", dev_samples.shape)

news_all.head()

news_all: (65238, 6)
dev_samples: (187009, 5)


,news_id,category,subcategory,title,abstract,text
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...","The Brands Queen Elizabeth, Prince Charles, an..."
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,50 Worst Habits For Belly Fat These seemingly ...
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,The Cost of Trump's Aid Freeze in the Trenches...
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",I Was An NBA Wife. Here's How It Affected My M...
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...","How to Get Rid of Skin Tags, According to a De..."


In [2]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    stop_words="english"
)

news_tfidf = vectorizer.fit_transform(news_all["text"])

news_id_to_idx = {
    news_id: idx
    for idx, news_id in enumerate(news_all["news_id"])
}

print(news_tfidf.shape)

(65238, 50000)


In [3]:
def get_history_indices(history, max_history=50):
    if pd.isna(history) or history == "":
        return []
    
    news_ids = history.split()[-max_history:]
    
    return [
        news_id_to_idx[nid]
        for nid in news_ids
        if nid in news_id_to_idx
    ]


def score_impression(group):
    history = group["history"].iloc[0]
    hist_indices = get_history_indices(history)
    
    group = group.copy()
    
    if len(hist_indices) == 0:
        group["score"] = 0.0
        return group
    
    user_vec = news_tfidf[hist_indices].mean(axis=0)
    
    scores = []
    for candidate_news in group["candidate_news"]:
        if candidate_news not in news_id_to_idx:
            scores.append(0.0)
            continue
        
        cand_idx = news_id_to_idx[candidate_news]
        cand_vec = news_tfidf[cand_idx]
        
        score = float(user_vec @ cand_vec.T)
        scores.append(score)
    
    group["score"] = scores
    return group

In [4]:
scored_groups = []

for _, group in tqdm(dev_samples.groupby("impression_id")):
    scored_groups.append(score_impression(group))

dev_scored = pd.concat(scored_groups, ignore_index=True)

print(dev_scored.shape)
dev_scored.head()

  0%|          | 0/5000 [00:00<?, ?it/s]

C:\Users\rlagu\AppData\Local\Temp\ipykernel_28732\232036816.py:35: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  score = float(user_vec @ cand_vec.T)
C:\Users\rlagu\AppData\Local\Temp\ipykernel_28732\232036816.py:35: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  score = float(user_vec @ cand_vec.T)
C:\Users\rlagu\AppData\Local\Temp\ipykernel_28732\232036816.py:35: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  score = float(user_vec @ cand_vec.T)
C:\Users\rlagu\AppD

(187009, 6)


,impression_id,user_id,history,candidate_news,label,score
0,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N28682,0,0.003380
1,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N48740,0,0.008121
2,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N31958,1,0.005173
3,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N34130,0,0.001394
4,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N6916,0,0.003035


In [5]:
save_path = OUTPUT_DIR / "tfidf_dev_scored.csv"

dev_scored.to_csv(save_path, index=False, encoding="utf-8-sig")

print("saved:", save_path)

saved: c:\Users\rlagu\Desktop\news-recommender-project\outputs\tfidf_dev_scored.csv


In [6]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

In [7]:
def mrr_score(labels, scores):
    labels = np.array(labels)
    scores = np.array(scores)
    
    order = np.argsort(scores)[::-1]
    sorted_labels = labels[order]
    
    positive_indices = np.where(sorted_labels == 1)[0]
    
    if len(positive_indices) == 0:
        return np.nan
    
    first_positive_rank = positive_indices[0] + 1
    return 1.0 / first_positive_rank


def dcg_at_k(labels, scores, k):
    labels = np.array(labels)
    scores = np.array(scores)
    
    order = np.argsort(scores)[::-1]
    sorted_labels = labels[order][:k]
    
    gains = sorted_labels
    discounts = np.log2(np.arange(2, len(sorted_labels) + 2))
    
    return np.sum(gains / discounts)


def ndcg_at_k(labels, scores, k):
    labels = np.array(labels)
    scores = np.array(scores)
    
    dcg = dcg_at_k(labels, scores, k)
    
    ideal_scores = labels
    idcg = dcg_at_k(labels, ideal_scores, k)
    
    if idcg == 0:
        return np.nan
    
    return dcg / idcg

In [8]:
metrics = []

for impression_id, group in dev_scored.groupby("impression_id"):
    labels = group["label"].values
    scores = group["score"].values
    
    # AUC는 label이 0과 1 둘 다 있어야 계산 가능
    if len(np.unique(labels)) < 2:
        auc = np.nan
    else:
        auc = roc_auc_score(labels, scores)
    
    mrr = mrr_score(labels, scores)
    ndcg5 = ndcg_at_k(labels, scores, k=5)
    ndcg10 = ndcg_at_k(labels, scores, k=10)
    
    metrics.append({
        "impression_id": impression_id,
        "AUC": auc,
        "MRR": mrr,
        "nDCG@5": ndcg5,
        "nDCG@10": ndcg10
    })

metrics_df = pd.DataFrame(metrics)

metrics_df.head()

,impression_id,AUC,MRR,nDCG@5,nDCG@10
0,1,0.857143,0.250000,0.430677,0.430677
1,2,0.166667,0.166667,0.000000,0.356207
2,3,0.159091,0.052632,0.000000,0.000000
3,4,0.520000,0.076923,0.000000,0.000000
4,5,0.416667,0.111111,0.000000,0.184576


In [9]:
tfidf_result = metrics_df[["AUC", "MRR", "nDCG@5", "nDCG@10"]].mean(skipna=True)

tfidf_result

AUC        0.577705
MRR        0.322725
nDCG@5     0.297889
nDCG@10    0.361070
dtype: float64

In [10]:
result_table = pd.DataFrame([
    {
        "model": "TF-IDF baseline",
        "AUC": tfidf_result["AUC"],
        "MRR": tfidf_result["MRR"],
        "nDCG@5": tfidf_result["nDCG@5"],
        "nDCG@10": tfidf_result["nDCG@10"]
    }
])

result_table

,model,AUC,MRR,nDCG@5,nDCG@10
0,TF-IDF baseline,0.577705,0.322725,0.297889,0.36107


In [11]:
result_table.to_csv(
    OUTPUT_DIR / "results.csv",
    index=False,
    encoding="utf-8-sig"
)

metrics_df.to_csv(
    OUTPUT_DIR / "tfidf_metrics_by_impression.csv",
    index=False,
    encoding="utf-8-sig"
)

print("saved!")

saved!


In [12]:
dev_scored[["label", "score"]].describe()

,label,score
count,187009.000000,187009.000000
mean,0.040618,0.004279
std,0.197405,0.007062
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.002608
75%,0.000000,0.005502
max,1.000000,0.437500


In [13]:
dev_scored.groupby("label")["score"].mean()

label
0    0.004175
1    0.006722
Name: score, dtype: float64